# Семинар 10. Python. Работа с текстами

## Мотивация

Данные исследователя почти всегда сначала приходят текстом: лог обучения, выгрузка
из чужой системы, csv от коллеги, ответ API. Пока текст не превращён в числа и
таблицы, анализировать нечего — а на этом пути ровно три типовые боли.

1. **Файл открывается «кракозябрами»** или падает с `UnicodeDecodeError`. Данные
   целы, но прочитаны не тем правилом — надо понимать, что такое кодировка.
2. **Нужные файлы разбросаны** по каталогам экспериментов: `runs/run-01/`,
   `runs/run-02/`, архив. Собрать их руками нельзя — нужна маска.
3. **Из строки лога надо достать значения**: время, номер эпохи, loss. Пока формат
   строго колоночный, хватает `split`; как только он «плавает», нужен более
   выразительный язык описания — регулярные выражения.

Сегодня разбираем эти три инструмента: кодировки (`bytes` и `str`), поиск файлов
по маске (`glob`, `pathlib`) и регулярные выражения (`re`). Домашка — парсер
текстового лога на регулярных выражениях, так что всё пригодится сразу.

В семинаре 3 похожие задачи решались командами оболочки: `grep`, `find`, `wc`.
Разница не в том, что «одно лучше другого»: командная строка удобна для быстрого
взгляда на файл, а Python сразу отдаёт результат объектом, который можно
посчитать, положить в таблицу и построить график.

## 1. Байты и кодировки

В файле и в сети нет «текста» — есть последовательность байт. **Кодировка** —
правило, по которому символ превращается в байты и обратно. В Python эти две
сущности разделены: `str` — последовательность символов, `bytes` —
последовательность байт. Перевод делают `encode` (символы → байты) и
`decode` (байты → символы).

```
  str  ──.encode("utf-8")──▶  bytes     символы превращаются в байты
  str  ◀──.decode("utf-8")──  bytes     байты разбираются обратно в символы
```

Новые файлы всегда пишем в UTF-8: это умолчание всего современного стека —
интернета, git, Python. `cp1251` и `koi8-r` нужны только чтобы **прочитать**
старую выгрузку, писать в них ничего нового не надо.

In [ ]:
text = "Привет, мир"
data = text.encode("utf-8")
print(len(text), len(data))
print(data)

Символов 11, а байт 20: в UTF-8 кириллическая буква занимает два байта,
а запятая и пробел — по одному. Поэтому длина строки и размер файла — разные
величины.

Кодировка UTF-8 не единственная. Старые выгрузки часто приходят в однобайтовой
`cp1251` (Windows-кириллица) или `koi8-r`.

In [ ]:
data_1251 = text.encode("cp1251")
print(len(data_1251), data_1251)
print(data_1251.decode("cp1251"))

Одному и тому же тексту соответствуют **разные** байты. Значит, байты без
указания кодировки прочитать нельзя — можно только угадать. Посмотрим, что
бывает при неверной догадке.

In [ ]:
broken = text.encode("utf-8").decode("cp1251")
print(broken)

Это и есть «кракозябры» (mojibake): файл не испорчен, верные байты прочитаны
неверным правилом. Такая порча обратима: раз каждому байту нашёлся символ, ни один байт не
потерялся — обратная пара `encode`/`decode` (в том же порядке кодировок)
вернёт исходный текст. Это и есть задача B1—B2 из `tasks.md`.

Обратная ошибка выглядит иначе.

In [ ]:
try:
    data_1251.decode("utf-8")
except UnicodeDecodeError as exc:
    print(exc)

#### ❓ **Вопрос**: Почему UTF-8-байты, прочитанные как cp1251, дали мусор, а cp1251-байты, прочитанные как UTF-8, — исключение?

<details>

<summary><strong>Ответ</strong></summary>

`cp1251` однобайтовая: у каждого из 256 байт есть свой символ, поэтому любая последовательность «читается» — просто не тем текстом.

`UTF-8` многобайтовая, и в ней у байт есть структура: первый байт многобайтового символа объявляет длину, продолжающие байты обязаны начинаться с `10`. Байты cp1251 этому не подчиняются, и `decode` сообщает об ошибке вместо тихой порчи данных.

</details>

Если данные важнее полноты, `decode` можно попросить не падать. Но это
осознанная потеря: аргумент `errors` подменяет или выбрасывает то, что не
разобралось.

In [ ]:
print(data_1251.decode("utf-8", errors="replace"))
print(data_1251.decode("utf-8", errors="ignore"))

`errors="replace"` ставит вместо каждого неразобранного байта символ-заменитель
U+FFFD — в Python он записывается как `"\ufffd"`, а в выводе выше это ромбики с
вопросительным знаком. `errors="ignore"` молча выбрасывает байты. Для разведки годится, для боевого разбора — нет:
испорченные строки должны быть видны, а не исчезать.

Теперь файлы. У `open`, `Path.read_text` и `Path.write_text` есть параметр
`encoding`. **Указывать его нужно явно.** Без него Python берёт кодировку
локали: на Linux это обычно UTF-8, а на Windows — ANSI-кодовая страница локали
(в русской это cp1251, в западноевропейской — cp1252). Один и тот же скрипт
ведёт себя на разных машинах по-разному.

In [ ]:
import locale
from pathlib import Path

work = Path("/tmp/course-text")
work.mkdir(exist_ok=True)
print(locale.getpreferredencoding(False))

In [ ]:
report = work / "report-1251.txt"
report.write_text("Модель обучена\n", encoding="cp1251")
print(report.read_bytes())
print(report.read_text(encoding="cp1251"), end="")

#### ❓ **Вопрос**: Файл записан в cp1251. Что произойдёт на Linux при чтении `report.read_text()` без `encoding`?

<details>

<summary><strong>Ответ</strong></summary>

Возьмётся кодировка локали — та, которую напечатала ячейка с `locale.getpreferredencoding` (на этой машине UTF-8; на другой в выводе может быть и `ANSI_X3.4-1968`). Байты cp1251 не раскладываются ни в UTF-8, ни в ASCII, поэтому будет `UnicodeDecodeError`, как в примере выше. А вот в русской Windows та же строка отработала бы без ошибки — результат зависит от машины, отсюда и правило указывать `encoding` явно.

</details>

Отдельная ловушка — **BOM** (byte order mark): три служебных байта в начале
файла, которыми Excel помечает UTF-8. Кодировка `utf-8-sig` их пишет и снимает,
а обычная `utf-8` о них не знает и отдаёт как невидимый символ.

In [ ]:
bom_file = work / "excel.csv"
bom_file.write_text("epoch,loss\n1,0.7100\n", encoding="utf-8-sig")
print(bom_file.read_bytes()[:3])
print(repr(bom_file.read_text(encoding="utf-8").split(",")[0]))
print(repr(bom_file.read_text(encoding="utf-8-sig").split(",")[0]))

#### ❓ **Вопрос**: csv из Excel прочитали как `utf-8`, и обращение к колонке `epoch` не работает, хотя в глаза колонка называется именно так. Что случилось?

<details>

<summary><strong>Ответ</strong></summary>

В начало имени попал невидимый символ `\ufeff` из BOM — это видно по `repr`: `'\ufeffepoch'` вместо `'epoch'`. Читать такой файл нужно с `encoding="utf-8-sig"`: тогда BOM снимается и имя становится обычным `'epoch'`.

</details>

## 2. Операции над строками

Примеры раздела — строки из того самого лога `train.log`, парсер которого
соберём в разделе 5: раздел 2 и есть его первый шаг.

Строки в Python **неизменяемы**: методы не правят строку на месте, а возвращают
новую. Поэтому разбор строки — это цепочка «отрезали → разделили → преобразовали».

In [ ]:
line = "  2026-11-03 12:04:12 INFO epoch=1 loss=0.7100  \n"
print(repr(line.strip()))
print(line.split())

`strip()` без аргументов снимает пробельные символы с обоих концов, включая
перевод строки, — это первое, что делают со строкой, прочитанной из файла.
`split()` без аргументов режет по любым пробелам и выбрасывает пустые куски.
Для csv так делать нельзя: там важны и разделитель, и пустые поля.

In [ ]:
print("1,,0.7100".split(","))
print("a b  c".split())
print("a b  c".split(" "))

`split(",")` сохранил пустое поле между запятыми, `split(" ")` — пустой кусок
между двумя пробелами. Разделять «по одному конкретному символу» и «по любым
пробелам» — разные задачи.

Пару «ключ = значение» удобнее разбирать через `partition`: он всегда возвращает
три части и не падает, если разделителя нет.

In [ ]:
key, sep, value = "loss=0.7100".partition("=")
print(repr(key), repr(value))
print("no-sign".partition("="))

Снять фиксированный префикс или суффикс нужно методами `removeprefix` и
`removesuffix` (Python 3.9+). Соблазн сделать это через `strip` — типичная и очень
живучая ошибка.

In [ ]:
name = "epoch_metrics.csv"
print(name.strip(".csv"))
print(name.removesuffix(".csv"))

#### ❓ **Вопрос**: Почему `"epoch_metrics.csv".strip(".csv")` вернул `epoch_metri`, а не `epoch_metrics`?

<details>

<summary><strong>Ответ</strong></summary>

`strip` принимает **набор символов**, а не подстроку. Набор здесь — `{'.', 'c', 's', 'v'}`, и с правого конца снимаются все символы из этого набора подряд: `v`, `s`, `c`, `.`, `s`, `c`. Остановка происходит на `i`, которого в наборе нет. Убрать именно суффикс умеет `removesuffix`.

</details>

Проверить префикс или суффикс, ничего не отрезая, умеют `startswith` и
`endswith`. Так отбирают файлы по расширению и отсеивают строки, не похожие на
начало записи лога.

In [ ]:
print(name.endswith(".csv"), name.startswith("epoch"))
print(line.startswith("2026-11-03"))
print(line.strip().startswith("2026-11-03"))

Второй вызов вернул `False`: в строке из файла впереди пробелы, и «начинается
с даты» ломается ровно на них — поэтому `strip()` делают первым делом. Оба
метода принимают и кортеж вариантов: `name.endswith((".csv", ".tsv"))`.

Важно помнить их границу: они сравнивают с **фиксированной** подстрокой. Как
только префикс переменный («строка начинается с любой даты»), нужен уже
`re.match` — им и займёмся в разделе 4.

Ещё две рабочие лошадки — `lower`/`upper` (регистр) и `replace` (заменить все
вхождения подстроки). Вместе с `removesuffix` они дают нормализацию имени: файлы,
названные по-разному, сводятся к одной форме и становятся сравнимы.

In [ ]:
raw_name = "RUN-05_2026-11-05.CSV"
print(raw_name.lower())
print(raw_name.lower().removesuffix(".csv").replace("-", "_"))

Обратная операция — сборка строки. `join` склеивает готовый список, f-строка
подставляет значения с нужным форматом.

In [ ]:
fields = ["2026-11-03", "1", "0.7100"]
print(",".join(fields))
loss = 0.71
print(f"epoch={1:03d} loss={loss:.2f}")

#### ❓ **Вопрос**: Строку `"loss=0.7100\n"` прочитали из файла и сразу применили `partition("=")`. Что окажется в значении и почему сначала нужен `strip()`?

<details>

<summary><strong>Ответ</strong></summary>

В значении окажется `'0.7100\n'` — вместе с переводом строки, потому что `partition` режет строку по разделителю и ничего не отбрасывает. Дальше `float('0.7100\n')` ещё сработает, а вот сравнение с `'0.7100'` или запись в csv уже дадут неожиданный результат. `strip()` снимает пробельные символы с концов, включая `\n`, — это видно по первой ячейке раздела, где `repr` показал строку до и после.

</details>

## 3. Поиск файлов по маске: glob

Эксперимент оставляет после себя дерево каталогов: `runs/run-01/`, `runs/run-02/`,
архив со старыми запусками. Задача «взять все `metrics.csv`» руками не решается —
нужна **маска имени** (glob-шаблон):

- `*` — любое число любых символов, кроме разделителя каталогов;
- `?` — ровно один символ;
- `[0-9]` — один символ из класса;
- `**` — любое число уровней вложенности (только с `recursive=True`).

Соберём учебное дерево. `iterdir` в последней строке — это «показать
содержимое каталога», один уровень, без всякой маски.

In [ ]:
runs = work / "runs"
for rel in ["run-01/metrics.csv", "run-02/metrics.csv",
            "run-02/notes.txt", "archive/run-03/metrics.csv"]:
    path = runs / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("epoch,loss\n", encoding="utf-8")
print(sorted(p.name for p in runs.iterdir()))

In [ ]:
import glob

print(glob.glob(str(runs / "*" / "metrics.csv")))

Один `*` соответствует ровно одному уровню вложенности, поэтому файл из
`archive/run-03/` не нашёлся. Рекурсивный поиск включает `**` вместе с
`recursive=True`.

In [ ]:
found = glob.glob(str(runs / "**" / "metrics.csv"), recursive=True)
print(sorted(found))

То же самое `pathlib` делает методами `glob` и `rglob` (`rglob` — рекурсивный
вариант). Результат — объекты `Path`, а не строки: у них сразу есть `.name`,
`.parent`, `.suffix`.

In [ ]:
for path in sorted(runs.rglob("metrics.csv")):
    print(path.relative_to(runs), path.parent.name)

Остальные шаблоны из шапки раздела работают так же, как в оболочке: `[0-9]` —
один символ из диапазона, `?` — ровно один любой символ.

In [ ]:
print(sorted(p.parent.name for p in runs.glob("run-0[1-9]/metrics.csv")))
print(sorted(runs.glob("run-?/metrics.csv")))
print(sorted(p.parent.name for p in runs.glob("run-??/metrics.csv")))

Маска `run-?` не нашла ничего: `?` — это **ровно один** символ, а в имени
`run-01` после дефиса их два, поэтому подходит `run-??`. Класс `[1-9]` полезен,
когда надо взять подмножество запусков, не перечисляя их имена.

#### ❓ **Вопрос**: Почему результат `glob` стоит оборачивать в `sorted`?

<details>

<summary><strong>Ответ</strong></summary>

Порядок выдачи не определён: он зависит от порядка записей в файловой системе. Пока порядок не зафиксирован, один и тот же скрипт на двух машинах даст разный порядок строк в отчёте, а diff покажет ложные изменения. `sorted` делает результат воспроизводимым.

</details>

Скрытые файлы (имя начинается с точки) два инструмента обрабатывают
по-разному. Положим такой файл в `runs/` и запросим обоими одну и ту же маску.

In [ ]:
(runs / ".hidden.csv").write_text("epoch,loss\n", encoding="utf-8")
print(glob.glob(str(runs / "*.csv")))
print([p.name for p in runs.glob("*.csv")])

#### ❓ **Вопрос**: Маска одна и та же, а `glob.glob` вернул пустой список, тогда как `Path.glob` нашёл `.hidden.csv`. Кто из них ошибается?

<details>

<summary><strong>Ответ</strong></summary>

Никто: это разное поведение по документации. `glob.glob` повторяет правило оболочки — `*` не совпадает с точкой в начале имени, скрытые файлы запрашивают явной маской `.*.csv`. У `pathlib` такого правила нет, и `Path.glob` возвращает скрытые файлы наравне с обычными.

Что выбирать: `glob.glob` удобен в однострочниках и когда нужны просто строки
путей (например, сразу отдать их в другую функцию); `pathlib` даёт объекты
`Path` с `.name`, `.suffix`, `.parent` — он выигрывает, как только с найденными
путями ещё работать. Устаревшим `glob` не является.

Практический вывод: переписывая код с одного инструмента на другой, результат можно молча изменить. Если скрытые файлы для задачи не нужны — отфильтруйте их сами, а не полагайтесь на умолчание.

</details>

## 4. Регулярные выражения

`split` хорош, пока формат строгий. Как только в строке появляется переменная
часть — необязательное поле, разное число пробелов, посторонние строки, — нужен
язык, описывающий **множество** подходящих строк. Это регулярные выражения; в
оболочке их использует `grep` (семинар 3), в Python — модуль `re`.

Основные элементы шаблона:

| Запись | Что значит |
|---|---|
| `\d` `\w` `\s` | цифра, буква/цифра/подчёркивание, пробельный символ |
| `.` | любой символ, кроме перевода строки |
| `+` `*` `?` | одно и больше, ноль и больше, ноль или одно |
| `{2,4}` | от двух до четырёх повторов |
| `[A-Z]` | один символ из класса |
| `^` `$` | начало и конец строки |
| `(...)` | группа: то, что нужно достать |
| `(?:...)` | группировка без захвата — в результат не попадает |
| `a\|b` | альтернатива: подходит либо `a`, либо `b` |
| `\S` `\W` `\D` | отрицание класса: не пробел, не буква/цифра, не цифра |
| `\b` | граница слова: ничего не съедает, но требует конца слова |

Шаблоны пишут **r-строками**: в обычной строке Python `\b` — это забой, а не
граница слова, и шаблон незаметно меняет смысл.

In [ ]:
import re

line = "2026-11-03 12:04:12 INFO epoch=1 loss=0.7100 acc=0.6210"
found = re.search(r"loss=(\d+\.\d+)", line)
print(found.group(0))
print(found.group(1))

`group(0)` — всё совпадение целиком, `group(1)` — первая группа в скобках,
то есть ровно то, что нужно достать. Если совпадения нет, `search` возвращает
`None` — проверять это обязательно.

У `re` три похожих функции поиска, и разница между ними — источник половины
ошибок новичков. Отличаются они только тем, где шаблону разрешено совпасть:

```
строка:      2026-11-03 12:04:12 INFO epoch=1 loss=0.7100
search:      ищет в любом месте               [epoch=1]  ✓
match:       только от начала строки  ^…                 ✗
fullmatch:   шаблон обязан покрыть всё  ^………………………$      ✗
```

In [ ]:
print(re.search(r"epoch=(\d+)", line).group(1))
print(re.match(r"epoch=(\d+)", line))
print(re.fullmatch(r"\d+", "42"))

#### ❓ **Вопрос**: `re.search` нашёл `epoch=1`, а `re.match` вернул `None` для того же шаблона и той же строки. Почему?

<details>

<summary><strong>Ответ</strong></summary>

`re.match` пытается совпасть **с начала строки**, а строка начинается с даты `2026-11-03`, а не с `epoch=`. `re.search` ищет совпадение в любом месте. Третья функция, `re.fullmatch`, требует, чтобы шаблон покрыл всю строку целиком, — она удобна для проверки формата.

</details>

В таблице выше остались невостребованными счётчик повторов `{n,m}` и класс
`[A-Z]`. Оба нужны, когда важна форма поля: дата — это ровно четыре цифры, дефис
и две пары цифр, а уровень лога — заглавные латинские буквы.

In [ ]:
head = "2026-11-03 12:04:12 WARNING nan detected in batch 17"
print(re.search(r"\d{4}-\d{2}-\d{2}", head).group(0))
print(re.findall(r"[A-Z]{2,}", head))

Заглавная буква инвертирует класс: `\S` — «любой непробельный символ», `\D` —
«любая не-цифра». А `\b` — граница слова: сама она ничего не совпадает, но требует,
чтобы слово в этом месте кончилось. Без неё шаблон `ERROR` найдётся и внутри
`ERRORS`.

In [ ]:
print(re.findall(r"\S+", "epoch=1  loss=0.7100"))
print(re.findall(r"ERROR", "ERRORS: ERROR at line 3"))
print(re.findall(r"ERROR\b", "ERRORS: ERROR at line 3"))

Последние две записи из таблицы — альтернатива `a|b` и не-захватывающая группа
`(?:...)`. Скобки в регулярке делают две разные вещи сразу: группируют и
захватывают. Когда нужна только группировка, пишут `(?:...)`.

In [ ]:
print(re.findall(r"(?:INFO|WARNING|ERROR)", head))
print(re.findall(r"(INFO|WARNING|ERROR) (\w+)", head))

#### ❓ **Вопрос**: Почему первый `findall` вернул само совпадение, а второй — кортеж?

<details>

<summary><strong>Ответ</strong></summary>

Если в шаблоне нет захватывающих групп, `findall` отдаёт текст самого совпадения (`'WARNING'`, а не всю строку): `(?:...)` только перечисляет варианты и в результат не попадает. Как только появились обычные скобки, `findall` переключается на содержимое групп — по кортежу на совпадение.

Отсюда практическое правило: скобки, поставленные «просто чтобы сгруппировать», молча меняют форму результата. Не нужен захват — пишите `(?:...)`.

</details>

Когда групп несколько, нумеровать их неудобно: добавили поле в середину — и
все номера уехали. Группам дают имена через `(?P<имя>...)`.

In [ ]:
pattern = re.compile(r"(?P<time>\S+ \S+) (?P<level>\w+) epoch=(?P<epoch>\d+)")
found = pattern.search(line)
print(found.group("level"), found.group("epoch"))
print(found.groupdict())

`re.compile` заранее разбирает шаблон. Разбирать строку шаблона на каждом вызове
`re.search` Python и так не станет: модуль `re` держит внутренний кеш последних
скомпилированных шаблонов. Но кеш ограничен несколькими сотнями записей и общий
на процесс, поэтому явный `compile` гарантирует однократный разбор, а главное —
шаблон получает имя и перестаёт дублироваться по коду.

Чтобы достать все совпадения, а не первое, есть `findall` и `finditer`.

In [ ]:
metrics = "loss=0.7100 acc=0.6210 loss=0.4213"
print(re.findall(r"(\w+)=([\d.]+)", metrics))
print(re.findall(r"\w+=", metrics))

`findall` с группами возвращает список кортежей (по одному на группу), без
групп — список строк. `finditer` вместо этого отдаёт объекты `Match`, у которых
есть и группы, и позиции — он нужен, когда важно, *где* нашлось.

In [ ]:
for found in re.finditer(r"(\w+)=([\d.]+)", metrics):
    print(found.group(1), found.start(), found.end())

`start()` и `end()` — позиции совпадения в исходной строке; по ним вырезают
контекст вокруг находки или подсвечивают её. Ещё одна функция того же ряда —
`re.split`: это `str.split`, у которого разделитель задан шаблоном.

In [ ]:
print(re.split(r"\s+", " epoch=1   loss=0.7100"))
print(" epoch=1   loss=0.7100".split())
print(re.split(r"[= ]", "epoch=1 loss=0.7100"))

`re.split` — не полный синоним `str.split()`: по краям он оставляет пустые куски
(ведущий пробел дал `''` в начале списка), а `str.split()` без аргументов их
отбрасывает. Зато шаблон умеет то, чего `str.split` не может: третий вызов режет
сразу по двум разным разделителям.

Следующая ловушка — **жадность** квантификаторов.

In [ ]:
markup = "<b>loss</b> <i>acc</i>"
print(re.findall(r"<.+>", markup))
print(re.findall(r"<.+?>", markup))

#### ❓ **Вопрос**: Почему `<.+>` вернул одно длинное совпадение, а `<.+?>` — четыре коротких?

<details>

<summary><strong>Ответ</strong></summary>

`+` жадный: он забирает максимум символов, при котором остаток шаблона всё ещё совпадает. `.` подходит и к `>`, поэтому совпадение растянулось от первого `<` до последнего `>`. Знак `?` после квантификатора делает его ленивым — берётся минимум, и каждый тег совпадает отдельно.

</details>

Регулярка умеет не только искать, но и заменять: `re.sub` подставляет вместо
совпадения новый текст, а `\1` в замене ссылается на группу.

In [ ]:
tail = "run finished user=ivanov token=s3cr3t-9f2a"
print(re.sub(r"token=\S+", "token=***", tail))
print(re.sub(r"user=(\w+)", r"user=<\1>", tail))

Поведение шаблона меняют **флаги**. Самый нужный при разборе логов —
`re.MULTILINE`: без него `^` и `$` — это начало и конец всего переданного текста,
а с ним — начало и конец каждой строки. Это важно, когда файл читают целиком, а
не построчно.

In [ ]:
log_text = "2026-11-03 INFO epoch=1\n2026-11-03 ERROR failed\n"
print(re.findall(r"^\S+ (\w+)", log_text))
print(re.findall(r"^\S+ (\w+)", log_text, re.MULTILINE))

#### ❓ **Вопрос**: Первый вызов нашёл только `INFO`, хотя строк в тексте две. Почему, и что изменил флаг?

<details>

<summary><strong>Ответ</strong></summary>

Без флагов `^` привязан к началу всего текста, поэтому совпадение возможно ровно одно — в самом начале, и это первая строка с `INFO`. `re.MULTILINE` заставляет `^` совпадать после каждого перевода строки, поэтому вторая строка с `ERROR` тоже попадает в результат.

Альтернатива — читать файл построчно через `splitlines()` и применять шаблон к каждой строке отдельно: тогда флаг не нужен.

</details>

Второй по частоте флаг — `re.IGNORECASE`. В логах разных систем уровень пишут и
`ERROR`, и `Error`, и `error`, и шаблон не должен от этого зависеть.

In [ ]:
print(re.findall(r"error", log_text))
print(re.findall(r"error", log_text, re.IGNORECASE))

Длинный шаблон нечитаем. Флаг `re.VERBOSE` разрешает переносы и комментарии
внутри шаблона — пробелы при этом игнорируются, а нужный пробел пишут как `\s`.

In [ ]:
row_re = re.compile(r"""
    epoch=(?P<epoch>\d+)     # номер эпохи
    \s+
    loss=(?P<loss>[\d.]+)     # значение функции потерь
""", re.VERBOSE)
print(row_re.search(line).groupdict())

## 5. Собираем парсер лога

Соберём инструменты вместе — ровно так устроена домашка. Записываем учебный лог: в
нём есть и содержательные строки, и посторонние. `splitlines()` разрезает текст на
список строк по переводам строки и сами `\n` в результат не кладёт.

In [ ]:
log_path = work / "train.log"
log_path.write_text("""2026-11-03 12:04:12 INFO epoch=1 loss=0.7100
2026-11-03 12:08:45 WARNING nan detected in batch 17
2026-11-03 12:08:46 INFO epoch=2 loss=0.4213
""", encoding="utf-8")
print(log_path.read_text(encoding="utf-8").splitlines()[1])

In [ ]:
rows = []
skipped = 0
for raw in log_path.read_text(encoding="utf-8").splitlines():
    found = row_re.search(raw)
    if found is None:
        skipped += 1
        continue
    rows.append((int(found["epoch"]), float(found["loss"])))

In [ ]:
print(rows)
print("нераспознанных строк:", skipped)

Тот же разбор, но без чтения файла целиком: файл открыт через `with`, а цикл идёт
по строкам. Так парсер переживёт лог любого размера.

In [ ]:
best = None
with open(log_path, encoding="utf-8") as handle:
    for raw in handle:
        found = row_re.search(raw)
        if found and (best is None or float(found["loss"]) < best):
            best = float(found["loss"])
print("лучший loss:", best)

Третья часть — поиск файлов. В домашке лог не один: их находят маской из
раздела 3, а разбор каждого остаётся тем же самым. Разложим наш лог по двум
запускам и пройдёмся по дереву.

In [ ]:
for name in ["run-01", "run-02"]:
    (runs / name / "train.log").write_text(
        log_path.read_text(encoding="utf-8"), encoding="utf-8")
for path in sorted(runs.rglob("train.log")):
    print(path.parent.name, len(row_re.findall(path.read_text(encoding="utf-8"))))

Схема разбора всегда одна: читаем построчно → пробуем шаблон → совпало
берём группы, не совпало считаем пропуск. `found["epoch"]` — короткая запись для
`found.group("epoch")`.

`read_text().splitlines()` загружает **весь файл в память** и делает из него
список строк. Для учебного лога это нормально, а для лога на сотни мегабайт —
нет: файловый объект сам по себе итератор по строкам, и память не растёт с
размером файла. Открывать файл при этом надо через `with`: он закроет файл и при
обычном выходе, и при исключении внутри цикла.

Считать пропущенные строки обязательно: иначе парсер молча теряет данные, и об
этом узнают уже по странному графику.

#### ❓ **Вопрос**: В лог добавилась строка `epoch=6 loss=nan`. Что сделает наш парсер и чем это опасно?

<details>

<summary><strong>Ответ</strong></summary>

Шаблон `[\d.]+` описывает только цифры и точку, а `nan` из букв — совпадения не будет, строка попадёт в `skipped`. Данные не испортятся, но эпоха тихо исчезнет из результата. Именно поэтому счётчик пропусков выводят рядом с результатом: ненулевое значение — сигнал посмотреть, какие строки не разобрались.

</details>

## Итог

- Текст в файле — байты; кодировка — правило перевода. `encoding` указываем явно,
  BOM снимаем через `utf-8-sig`.
- Методы `str` разбирают строгий формат: `strip`, `split`, `partition`,
  `removeprefix`/`removesuffix`. `strip` работает с набором символов, а не с
  подстрокой.
- `glob` и `Path.rglob` собирают файлы по маске; результат сортируем.
- `re` описывает изменчивый формат: группы (лучше именованные), `search` против
  `match`, жадность, `sub` для замены, `VERBOSE` для читаемости.

Домашка — парсер текстового лога на регулярных выражениях. Задачи семинара — в
`tasks.md`, данные готовит `python3 assets/make_data.py`.